In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta del Día 10.
CARPETA_DATOS = Path("datos_dia_10")
CARPETA_DATOS.mkdir(exist_ok=True)

# Base de datos que se generará a partir del diseño ER.
RUTA_DB = CARPETA_DATOS / "ventas_desde_diagrama_er.db"

# Archivo Markdown donde guardaremos el diagrama Mermaid.
RUTA_DIAGRAMA = CARPETA_DATOS / "diagrama_er_ventas.md"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

print("\nArchivo del diagrama:")
print(RUTA_DIAGRAMA.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_10

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_10\ventas_desde_diagrama_er.db

Archivo del diagrama:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_10\diagrama_er_ventas.md


In [2]:
entidades = pd.DataFrame([
    {
        "entidad": "clientes",
        "descripcion": "Personas que realizan compras",
        "pregunta_que_responde": "¿Quién compra?"
    },
    {
        "entidad": "productos",
        "descripcion": "Artículos disponibles para venta",
        "pregunta_que_responde": "¿Qué se vende?"
    },
    {
        "entidad": "ventas",
        "descripcion": "Encabezado de una operación de venta",
        "pregunta_que_responde": "¿Cuándo se vendió y a qué cliente?"
    },
    {
        "entidad": "detalle_ventas",
        "descripcion": "Productos incluidos en cada venta",
        "pregunta_que_responde": "¿Qué productos se vendieron en cada venta?"
    }
])

entidades

,entidad,descripcion,pregunta_que_responde
0,clientes,Personas que realizan compras,¿Quién compra?
1,productos,Artículos disponibles para venta,¿Qué se vende?
2,ventas,Encabezado de una operación de venta,¿Cuándo se vendió y a qué cliente?
3,detalle_ventas,Productos incluidos en cada venta,¿Qué productos se vendieron en cada venta?


In [3]:
atributos = pd.DataFrame([
    {"tabla": "clientes", "atributo": "id_cliente", "tipo": "INTEGER", "rol": "PK"},
    {"tabla": "clientes", "atributo": "nombre", "tipo": "TEXT", "rol": "Dato"},
    {"tabla": "clientes", "atributo": "correo", "tipo": "TEXT", "rol": "Dato único"},
    {"tabla": "clientes", "atributo": "telefono", "tipo": "TEXT", "rol": "Dato opcional"},
    {"tabla": "clientes", "atributo": "ciudad", "tipo": "TEXT", "rol": "Dato"},

    {"tabla": "productos", "atributo": "id_producto", "tipo": "INTEGER", "rol": "PK"},
    {"tabla": "productos", "atributo": "sku", "tipo": "TEXT", "rol": "Dato único"},
    {"tabla": "productos", "atributo": "nombre", "tipo": "TEXT", "rol": "Dato"},
    {"tabla": "productos", "atributo": "categoria", "tipo": "TEXT", "rol": "Dato"},
    {"tabla": "productos", "atributo": "precio", "tipo": "REAL", "rol": "Dato validado"},
    {"tabla": "productos", "atributo": "stock", "tipo": "INTEGER", "rol": "Dato validado"},
    {"tabla": "productos", "atributo": "activo", "tipo": "INTEGER", "rol": "Dato de control"},

    {"tabla": "ventas", "atributo": "id_venta", "tipo": "INTEGER", "rol": "PK"},
    {"tabla": "ventas", "atributo": "id_cliente", "tipo": "INTEGER", "rol": "FK"},
    {"tabla": "ventas", "atributo": "fecha", "tipo": "TEXT", "rol": "Dato"},
    {"tabla": "ventas", "atributo": "total", "tipo": "REAL", "rol": "Dato calculado o almacenado"},

    {"tabla": "detalle_ventas", "atributo": "id_detalle", "tipo": "INTEGER", "rol": "PK"},
    {"tabla": "detalle_ventas", "atributo": "id_venta", "tipo": "INTEGER", "rol": "FK"},
    {"tabla": "detalle_ventas", "atributo": "id_producto", "tipo": "INTEGER", "rol": "FK"},
    {"tabla": "detalle_ventas", "atributo": "cantidad", "tipo": "INTEGER", "rol": "Dato validado"},
    {"tabla": "detalle_ventas", "atributo": "precio_unitario", "tipo": "REAL", "rol": "Precio histórico"},
    {"tabla": "detalle_ventas", "atributo": "subtotal", "tipo": "REAL", "rol": "Dato calculado"}
])

atributos

,tabla,atributo,tipo,rol
0,clientes,id_cliente,INTEGER,PK
1,clientes,nombre,TEXT,Dato
2,clientes,correo,TEXT,Dato único
3,clientes,telefono,TEXT,Dato opcional
4,clientes,ciudad,TEXT,Dato
5,productos,id_producto,INTEGER,PK
6,productos,sku,TEXT,Dato único
7,productos,nombre,TEXT,Dato
8,productos,categoria,TEXT,Dato
9,productos,precio,REAL,Dato validado


In [4]:
relaciones = pd.DataFrame([
    {
        "tabla_padre": "clientes",
        "tabla_hija": "ventas",
        "cardinalidad": "1:N",
        "campo_relacion": "ventas.id_cliente -> clientes.id_cliente",
        "explicacion": "Un cliente puede tener muchas ventas, pero cada venta pertenece a un cliente."
    },
    {
        "tabla_padre": "ventas",
        "tabla_hija": "detalle_ventas",
        "cardinalidad": "1:N",
        "campo_relacion": "detalle_ventas.id_venta -> ventas.id_venta",
        "explicacion": "Una venta puede tener muchos renglones de detalle."
    },
    {
        "tabla_padre": "productos",
        "tabla_hija": "detalle_ventas",
        "cardinalidad": "1:N",
        "campo_relacion": "detalle_ventas.id_producto -> productos.id_producto",
        "explicacion": "Un producto puede aparecer en muchos detalles de venta."
    }
])

relaciones

,tabla_padre,tabla_hija,cardinalidad,campo_relacion,explicacion
0,clientes,ventas,1:N,ventas.id_cliente -> clientes.id_cliente,"Un cliente puede tener muchas ventas, pero cad..."
1,ventas,detalle_ventas,1:N,detalle_ventas.id_venta -> ventas.id_venta,Una venta puede tener muchos renglones de deta...
2,productos,detalle_ventas,1:N,detalle_ventas.id_producto -> productos.id_pro...,Un producto puede aparecer en muchos detalles ...


In [5]:
diagrama_mermaid = """
erDiagram
    CLIENTES ||--o{ VENTAS : realiza
    VENTAS ||--|{ DETALLE_VENTAS : contiene
    PRODUCTOS ||--o{ DETALLE_VENTAS : aparece_en

    CLIENTES {
        INTEGER id_cliente PK
        TEXT nombre
        TEXT correo
        TEXT telefono
        TEXT ciudad
    }

    PRODUCTOS {
        INTEGER id_producto PK
        TEXT sku
        TEXT nombre
        TEXT categoria
        REAL precio
        INTEGER stock
        INTEGER activo
    }

    VENTAS {
        INTEGER id_venta PK
        INTEGER id_cliente FK
        TEXT fecha
        REAL total
    }

    DETALLE_VENTAS {
        INTEGER id_detalle PK
        INTEGER id_venta FK
        INTEGER id_producto FK
        INTEGER cantidad
        REAL precio_unitario
        REAL subtotal
    }
"""
print(diagrama_mermaid)


erDiagram
    CLIENTES ||--o{ VENTAS : realiza
    VENTAS ||--|{ DETALLE_VENTAS : contiene
    PRODUCTOS ||--o{ DETALLE_VENTAS : aparece_en

    CLIENTES {
        INTEGER id_cliente PK
        TEXT nombre
        TEXT correo
        TEXT telefono
        TEXT ciudad
    }

    PRODUCTOS {
        INTEGER id_producto PK
        TEXT sku
        TEXT nombre
        TEXT categoria
        REAL precio
        INTEGER stock
        INTEGER activo
    }

    VENTAS {
        INTEGER id_venta PK
        INTEGER id_cliente FK
        TEXT fecha
        REAL total
    }

    DETALLE_VENTAS {
        INTEGER id_detalle PK
        INTEGER id_venta FK
        INTEGER id_producto FK
        INTEGER cantidad
        REAL precio_unitario
        REAL subtotal
    }



In [7]:
def obtener_conexion():
    conexion = sqlite3.connect(RUTA_DB)
    conexion.execute("PRAGMA foreign_keys = ON")
    return conexion


conexion = obtener_conexion()

estado_fk = pd.read_sql_query("""
PRAGMA foreign_keys
""", conexion)

conexion.close()

estado_fk

,foreign_keys
0,1


In [8]:
conexion = obtener_conexion()
cursor = conexion.cursor()

# Primero eliminamos tablas hijas y después tablas padre.
cursor.execute("DROP TABLE IF EXISTS detalle_ventas")
cursor.execute("DROP TABLE IF EXISTS ventas")
cursor.execute("DROP TABLE IF EXISTS productos")
cursor.execute("DROP TABLE IF EXISTS clientes")

cursor.execute("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    ciudad TEXT DEFAULT 'No especificada'
)
""")

cursor.execute("""
CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
)
""")

cursor.execute("""
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    fecha TEXT NOT NULL,
    total REAL NOT NULL DEFAULT 0 CHECK(total >= 0),

    FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
)
""")

cursor.execute("""
CREATE TABLE detalle_ventas (
    id_detalle INTEGER PRIMARY KEY,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0),
    subtotal REAL NOT NULL CHECK(subtotal >= 0),

    FOREIGN KEY (id_venta)
        REFERENCES ventas(id_venta)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (id_producto)
        REFERENCES productos(id_producto)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
)
""")

conexion.commit()
conexion.close()

print("Tablas creadas correctamente a partir del diagrama ER.")

Tablas creadas correctamente a partir del diagrama ER.


In [9]:
conexion = obtener_conexion()

tablas_creadas = pd.read_sql_query("""
SELECT name AS tabla
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", conexion)

conexion.close()

tablas_creadas

,tabla
0,clientes
1,detalle_ventas
2,productos
3,ventas


In [10]:
conexion = obtener_conexion()

for tabla in ["ventas", "detalle_ventas"]:
    print(f"Llaves foráneas de la tabla: {tabla}")
    display(pd.read_sql_query(f"""
    PRAGMA foreign_key_list({tabla})
    """, conexion))

conexion.close()

Llaves foráneas de la tabla: ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,clientes,id_cliente,id_cliente,CASCADE,RESTRICT,NONE


Llaves foráneas de la tabla: detalle_ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,productos,id_producto,id_producto,CASCADE,RESTRICT,NONE
1,1,0,ventas,id_venta,id_venta,CASCADE,RESTRICT,NONE


In [11]:
conexion = obtener_conexion()
cursor = conexion.cursor()

clientes = [
    (1, "Ana López", "ana@example.com", "555-111-2222", "CDMX"),
    (2, "Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara"),
    (3, "María Torres", "maria@example.com", None, "Monterrey"),
    (4, "Luis Romero", "luis@example.com", "555-888-9999", "Puebla")
]

productos = [
    (1, "P001", "Mouse inalámbrico", "Accesorios", 249.90, 15, 1),
    (2, "P002", "Teclado mecánico", "Accesorios", 899.00, 8, 1),
    (3, "P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 4, 1),
    (4, "P004", "Cable HDMI", "Cables", 120.00, 20, 1),
    (5, "P005", "Memoria USB 64GB", "Almacenamiento", 150.00, 30, 1)
]

cursor.executemany("""
INSERT INTO clientes (id_cliente, nombre, correo, telefono, ciudad)
VALUES (?, ?, ?, ?, ?)
""", clientes)

cursor.executemany("""
INSERT INTO productos (id_producto, sku, nombre, categoria, precio, stock, activo)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", productos)

conexion.commit()
conexion.close()

print("Clientes y productos insertados correctamente.")

Clientes y productos insertados correctamente.


In [12]:
conexion = obtener_conexion()
cursor = conexion.cursor()

ventas = [
    (1, 1, "2026-06-28", 1398.80),
    (2, 2, "2026-06-29", 3539.00),
    (3, 1, "2026-06-30", 570.00)
]

detalle_ventas = [
    (1, 1, 1, 2, 249.90, 499.80),
    (2, 1, 2, 1, 899.00, 899.00),
    (3, 2, 3, 1, 3299.00, 3299.00),
    (4, 2, 4, 2, 120.00, 240.00),
    (5, 3, 4, 3, 120.00, 360.00),
    (6, 3, 5, 2, 150.00, 300.00)
]

cursor.executemany("""
INSERT INTO ventas (id_venta, id_cliente, fecha, total)
VALUES (?, ?, ?, ?)
""", ventas)

cursor.executemany("""
INSERT INTO detalle_ventas (
    id_detalle,
    id_venta,
    id_producto,
    cantidad,
    precio_unitario,
    subtotal
)
VALUES (?, ?, ?, ?, ?, ?)
""", detalle_ventas)

conexion.commit()
conexion.close()

print("Ventas y detalles insertados correctamente.")

Ventas y detalles insertados correctamente.


In [13]:
conexion = obtener_conexion()

print("Clientes")
display(pd.read_sql_query("SELECT * FROM clientes", conexion))

print("Productos")
display(pd.read_sql_query("SELECT * FROM productos", conexion))

print("Ventas")
display(pd.read_sql_query("SELECT * FROM ventas", conexion))

print("Detalle de ventas")
display(pd.read_sql_query("SELECT * FROM detalle_ventas", conexion))

conexion.close()

Clientes


,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey
3,4,Luis Romero,luis@example.com,555-888-9999,Puebla


Productos


,id_producto,sku,nombre,categoria,precio,stock,activo
0,1,P001,Mouse inalámbrico,Accesorios,249.9,15,1
1,2,P002,Teclado mecánico,Accesorios,899.0,8,1
2,3,P003,Monitor 24 pulgadas,Pantallas,3299.0,4,1
3,4,P004,Cable HDMI,Cables,120.0,20,1
4,5,P005,Memoria USB 64GB,Almacenamiento,150.0,30,1


Ventas


,id_venta,id_cliente,fecha,total
0,1,1,2026-06-28,1398.8
1,2,2,2026-06-29,3539.0
2,3,1,2026-06-30,570.0


Detalle de ventas


,id_detalle,id_venta,id_producto,cantidad,precio_unitario,subtotal
0,1,1,1,2,249.9,499.8
1,2,1,2,1,899.0,899.0
2,3,2,3,1,3299.0,3299.0
3,4,2,4,2,120.0,240.0
4,5,3,4,3,120.0,360.0
5,6,3,5,2,150.0,300.0


In [14]:
conexion = obtener_conexion()

reporte_ventas = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.correo,
    p.sku,
    p.nombre AS producto,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal,
    v.total AS total_venta
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

reporte_ventas

,id_venta,fecha,cliente,correo,sku,producto,cantidad,precio_unitario,subtotal,total_venta
0,1,2026-06-28,Ana López,ana@example.com,P001,Mouse inalámbrico,2,249.9,499.8,1398.8
1,1,2026-06-28,Ana López,ana@example.com,P002,Teclado mecánico,1,899.0,899.0,1398.8
2,2,2026-06-29,Carlos Pérez,carlos@example.com,P003,Monitor 24 pulgadas,1,3299.0,3299.0,3539.0
3,2,2026-06-29,Carlos Pérez,carlos@example.com,P004,Cable HDMI,2,120.0,240.0,3539.0
4,3,2026-06-30,Ana López,ana@example.com,P004,Cable HDMI,3,120.0,360.0,570.0
5,3,2026-06-30,Ana López,ana@example.com,P005,Memoria USB 64GB,2,150.0,300.0,570.0


In [15]:
conexion = obtener_conexion()

productos_por_venta = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    p.nombre AS producto,
    dv.cantidad
FROM ventas v
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, p.nombre
""", conexion)

conexion.close()

productos_por_venta

,id_venta,fecha,producto,cantidad
0,1,2026-06-28,Mouse inalámbrico,2
1,1,2026-06-28,Teclado mecánico,1
2,2,2026-06-29,Cable HDMI,2
3,2,2026-06-29,Monitor 24 pulgadas,1
4,3,2026-06-30,Cable HDMI,3
5,3,2026-06-30,Memoria USB 64GB,2


In [16]:
conexion = obtener_conexion()

ventas_por_producto = pd.read_sql_query("""
SELECT
    p.id_producto,
    p.nombre AS producto,
    v.id_venta,
    v.fecha,
    dv.cantidad
FROM productos p
INNER JOIN detalle_ventas dv
    ON p.id_producto = dv.id_producto
INNER JOIN ventas v
    ON dv.id_venta = v.id_venta
ORDER BY p.id_producto, v.id_venta
""", conexion)

conexion.close()

ventas_por_producto

,id_producto,producto,id_venta,fecha,cantidad
0,1,Mouse inalámbrico,1,2026-06-28,2
1,2,Teclado mecánico,1,2026-06-28,1
2,3,Monitor 24 pulgadas,2,2026-06-29,1
3,4,Cable HDMI,2,2026-06-29,2
4,4,Cable HDMI,3,2026-06-30,3
5,5,Memoria USB 64GB,3,2026-06-30,2


In [17]:
conexion = obtener_conexion()

base_matriz = pd.read_sql_query("""
SELECT
    v.id_venta,
    p.nombre AS producto,
    dv.cantidad
FROM ventas v
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
""", conexion)

conexion.close()

matriz_venta_producto = base_matriz.pivot_table(
    index="id_venta",
    columns="producto",
    values="cantidad",
    aggfunc="sum",
    fill_value=0
)

matriz_venta_producto

producto,Cable HDMI,Memoria USB 64GB,Monitor 24 pulgadas,Mouse inalámbrico,Teclado mecánico
id_venta,,,,,
1,0,0,0,2,1
2,2,0,1,0,0
3,3,2,0,0,0


In [18]:
conexion = obtener_conexion()
cursor = conexion.cursor()

try:
    cursor.execute("""
    INSERT INTO ventas (id_venta, id_cliente, fecha, total)
    VALUES (?, ?, ?, ?)
    """, (99, 999, "2026-07-01", 0))

    conexion.commit()
    print("Venta insertada.")

except sqlite3.IntegrityError as error:
    print("No se pudo insertar la venta.")
    print("Motivo:", error)

finally:
    conexion.close()

No se pudo insertar la venta.
Motivo: FOREIGN KEY constraint failed


In [19]:
def obtener_tablas(conexion):
    resultado = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """, conexion)
    
    return set(resultado["name"])


def obtener_columnas(conexion, tabla):
    resultado = pd.read_sql_query(f"""
    PRAGMA table_info({tabla})
    """, conexion)
    
    return set(resultado["name"])


def obtener_llaves_foraneas(conexion, tabla):
    resultado = pd.read_sql_query(f"""
    PRAGMA foreign_key_list({tabla})
    """, conexion)
    
    return resultado


conexion = obtener_conexion()

tablas_esperadas = {"clientes", "productos", "ventas", "detalle_ventas"}
tablas_existentes = obtener_tablas(conexion)

print("Tablas esperadas:")
print(tablas_esperadas)

print("\nTablas existentes:")
print(tablas_existentes)

print("\n¿Están todas las tablas esperadas?")
print(tablas_esperadas.issubset(tablas_existentes))

print("\nColumnas de clientes:")
print(obtener_columnas(conexion, "clientes"))

print("\nColumnas de productos:")
print(obtener_columnas(conexion, "productos"))

print("\nColumnas de ventas:")
print(obtener_columnas(conexion, "ventas"))

print("\nColumnas de detalle_ventas:")
print(obtener_columnas(conexion, "detalle_ventas"))

print("\nLlaves foráneas de ventas:")
display(obtener_llaves_foraneas(conexion, "ventas"))

print("\nLlaves foráneas de detalle_ventas:")
display(obtener_llaves_foraneas(conexion, "detalle_ventas"))

conexion.close()

Tablas esperadas:
{'clientes', 'detalle_ventas', 'ventas', 'productos'}

Tablas existentes:
{'clientes', 'detalle_ventas', 'ventas', 'productos'}

¿Están todas las tablas esperadas?
True

Columnas de clientes:
{'correo', 'nombre', 'id_cliente', 'telefono', 'ciudad'}

Columnas de productos:
{'precio', 'categoria', 'nombre', 'id_producto', 'sku', 'stock', 'activo'}

Columnas de ventas:
{'fecha', 'id_cliente', 'id_venta', 'total'}

Columnas de detalle_ventas:
{'id_venta', 'subtotal', 'precio_unitario', 'id_detalle', 'id_producto', 'cantidad'}

Llaves foráneas de ventas:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,clientes,id_cliente,id_cliente,CASCADE,RESTRICT,NONE



Llaves foráneas de detalle_ventas:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,productos,id_producto,id_producto,CASCADE,RESTRICT,NONE
1,1,0,ventas,id_venta,id_venta,CASCADE,RESTRICT,NONE
